## UrbanRide_11 — Churn Prediction Model
**Type:** Binary Classification  
**Label:** `is_churned` (true = churned, false = active)  
**Source:** `urbanride.gold.customer_features`  
**Schedule:** Weekly · Sunday 04:00 AM (Job 4)

**Models trained:**
- Logistic Regression — baseline
- Random Forest — main model

**Class imbalance:** 12.1% churned vs 87.9% active — handled via `weightCol`  
**Split strategy:** Time-based on `signup_date` at Jan 1 2026 — 67% train / 33% test — prevents data leakage  
**Experiment:** `/urbanride_churn_prediction`  
**Registry:** `urbanride_churn_model`

**Key features:**
- `days_since_last_trip` — strongest churn signal (9 days active vs 72 days churned)
- `total_trips`, `total_spend`, `customer_age_days` — strong signals
- `city`, `device_type`, `referral_source`, `favourite_vehicle_type` — encoded categoricals
- `preferred_payment_method` dropped — zero variance (all UPI)

**Fixes applied:**
- TMP_DIR points to silver Volume `mlflow_artifacts` — create this Volume in Catalog UI before running
- Feature importance uses `tempfile.TemporaryDirectory()` — no hardcoded paths
- RF loop uses `del rf_model` + `del best_rf_model` + `gc.collect()` — prevents 1GB cache overflow
- Cell 0 added — cleans Python memory before every run

**Output:** Predictions written to `urbanride.gold.churn_predictions`  
**Feeds:** Customer Segmentation Model (14)

In [0]:
# ── Run this cell first before anything else ─────────────────
# Clears stale ML models from previous sessions (ecommerce challenge etc.)
# Prevents 1GB ML cache overflow error on serverless

import gc

# Delete any stale model objects from previous sessions
stale_models = [
    'lr_model', 'rf_model', 'best_rf_model',
    'BEST_MODEL', 'rf_pipeline', 'lr_pipeline'
]
for name in stale_models:
    try:
        del globals()[name]
        print(f'  Deleted: {name}')
    except KeyError:
        pass  # not in memory — skip

# Force Python garbage collection
gc.collect()
print('  Python memory cleared.')
print()

# Verify Spark is responsive
spark.sql('SELECT 1').show()
print('  Spark connection healthy.')

# Verify catalog is accessible
spark.sql('SHOW SCHEMAS IN urbanride').show()
print('  Catalog accessible.')

print()
print('Session ready. Safe to run notebook.')


In [0]:
import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when, lit, current_timestamp
import pandas as pd
import gc
import tempfile
import os

# Do NOT call mlflow.autolog() — crashes on serverless

CATALOG         = 'urbanride'
GOLD            = f'{CATALOG}.gold'
EXPERIMENT_NAME = '/urbanride_churn_prediction'
MODEL_NAME      = 'urbanride_churn_model'

# TMP_DIR — MLflow uses this to stage model files during registration
# PREREQUISITE: Create Volume 'mlflow_artifacts' in silver schema from Catalog UI first
# Catalog → urbanride → silver → Create → Create Volume → name: mlflow_artifacts
TMP_DIR = '/Volumes/urbanride/silver/mlflow_artifacts/mlflow_tmp'
dbutils.fs.mkdirs(TMP_DIR)
print(f'TMP_DIR ready: {TMP_DIR}')

# Feature columns
# preferred_payment_method excluded — zero variance (all UPI)
NUMERIC_COLS = [
    'days_since_last_trip',
    'total_trips',
    'completed_trips',
    'cancelled_trips',
    'cancellation_rate',
    'total_spend',
    'avg_spend_per_trip',
    'customer_age_days',
    'avg_trip_distance_km',
    'avg_trip_duration_minutes',
    'avg_surge_multiplier',
    'weekend_trip_ratio',
    'rainy_day_trip_ratio',
    'failed_payment_count',
    'failed_payment_rate',
]

CATEGORICAL_COLS = [
    'city',
    'device_type',
    'referral_source',
    'favourite_vehicle_type',
]

LABEL_COL  = 'label'
WEIGHT_COL = 'class_weight'

# Set MLflow experiment
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Catalog             : {CATALOG}')
print(f'Source              : {GOLD}.customer_features')
print(f'Experiment          : {EXPERIMENT_NAME}')
print(f'Model name          : {MODEL_NAME}')
print(f'Numeric features    : {len(NUMERIC_COLS)}')
print(f'Categorical features: {len(CATEGORICAL_COLS)}')
print('Config ready.')


In [0]:
print('Loading gold.customer_features...')

df_raw = spark.table(f'{GOLD}.customer_features')

print(f'  Total rows : {df_raw.count():,}')
print()

# Churn distribution
print('Churn distribution:')
df_raw.groupBy('is_churned').count().orderBy('is_churned').show()

# Select only needed columns
ALL_FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS

df = df_raw.select(
    ['customer_id', 'signup_date', 'is_churned'] + ALL_FEATURE_COLS
).withColumn(
    # Convert boolean label to integer — Spark ML requires numeric labels
    # true  → 1 (churned)
    # false → 0 (active)
    LABEL_COL,
    col('is_churned').cast('int')
)

# Drop rows with null label
df = df.filter(col(LABEL_COL).isNotNull())

print(f'  Rows after null label filter : {df.count():,}')


In [0]:
# Time-based split on signup_date
# Train : customers who signed up before Jan 1 2026  (~67%)
# Test  : customers who signed up on or after Jan 1 2026 (~33%)
#
# Why time-based not random?
# Prevents data leakage — model is always trained on past customers
# and tested on future customers, which is how it will work in production
#
# Why Jan 1 2026?
# Data runs Sep 2025 → Mar 2026 (182 days)
# Jan 1 gives ~4 months train, ~2 months test — standard 67/33 split

from pyspark.sql.functions import to_date

SPLIT_DATE = '2026-01-01'

df_train = df.filter(col('signup_date') <  SPLIT_DATE)
df_test  = df.filter(col('signup_date') >= SPLIT_DATE)

train_count = df_train.count()
test_count  = df_test.count()
total       = train_count + test_count

print(f'Split date : {SPLIT_DATE}')
print(f'Train rows : {train_count:,} ({train_count/total*100:.1f}%)')
print(f'Test rows  : {test_count:,}  ({test_count/total*100:.1f}%)')
print()

# Sanity check — churn rate should be similar in both splits
# Train will have higher churn rate than test — expected
# because train customers had more time to churn (older signups)
print('Churn rate in train set:')
df_train.groupBy('is_churned').count().orderBy('is_churned').show()
print('Churn rate in test set:')
df_test.groupBy('is_churned').count().orderBy('is_churned').show()


In [0]:
# Class imbalance: ~12% churned vs ~88% active
# Without class weights the model predicts 'active' for everything
# and gets 88% accuracy while missing every single churned customer
#
# Class weight formula: weight = total / (num_classes * class_count)
# Churned customers are rare → higher weight → model pays more attention to them
# Active customers are common → lower weight → model doesn't over-optimise for them

print('Computing class weights...')

train_total   = df_train.count()
churned_count = df_train.filter(col(LABEL_COL) == 1).count()
active_count  = df_train.filter(col(LABEL_COL) == 0).count()
num_classes   = 2

weight_churned = train_total / (num_classes * churned_count)
weight_active  = train_total / (num_classes * active_count)

print(f'  Train total    : {train_total:,}')
print(f'  Churned count  : {churned_count:,}')
print(f'  Active count   : {active_count:,}')
print(f'  Weight churned : {weight_churned:.4f}')
print(f'  Weight active  : {weight_active:.4f}')

# Add class weight column to train set
df_train = df_train.withColumn(
    WEIGHT_COL,
    when(col(LABEL_COL) == 1, weight_churned)
    .otherwise(weight_active)
)

# Test set does not need weights — weights only used during training
df_test = df_test.withColumn(WEIGHT_COL, lit(1.0))

print('Class weights added to train set.')


In [0]:
print('Building feature pipeline...')

# Step 1: StringIndexer — convert string categories to numbers
# e.g. city: 'Delhi'=0, 'Mumbai'=1, 'Bengaluru'=2 etc.
# handleInvalid='keep' — unseen categories in test set get a new index instead of crashing
indexers = [
    StringIndexer(
        inputCol     = c,
        outputCol    = f'{c}_idx',
        handleInvalid= 'keep'
    )
    for c in CATEGORICAL_COLS
]

# Step 2: VectorAssembler — combine all features into one vector column
# Spark ML requires all features in a single 'features' vector column
indexed_cat_cols = [f'{c}_idx' for c in CATEGORICAL_COLS]
ALL_INPUT_COLS   = NUMERIC_COLS + indexed_cat_cols

assembler = VectorAssembler(
    inputCols    = ALL_INPUT_COLS,
    outputCol    = 'features_raw',
    handleInvalid= 'skip'
)

# Step 3: StandardScaler — normalise feature ranges
# Needed for Logistic Regression only
# total_spend (~30,000) vs cancellation_rate (~0.08) — huge difference in range
# Without scaling LR gets confused by magnitude — treats large numbers as more important
# withMean=True, withStd=True — centres and scales each feature
# RF does NOT use this — trees are scale-invariant
scaler = StandardScaler(
    inputCol = 'features_raw',
    outputCol= 'features',
    withMean = True,
    withStd  = True
)

print(f'  Numeric features    : {len(NUMERIC_COLS)}')
print(f'  Categorical features: {len(CATEGORICAL_COLS)} → {len(indexed_cat_cols)} indexed')
print(f'  Total input cols    : {len(ALL_INPUT_COLS)}')
print('Feature pipeline ready.')


In [0]:
print('Training Logistic Regression (baseline)...')

lr = LogisticRegression(
    featuresCol = 'features',
    labelCol    = LABEL_COL,
    weightCol   = WEIGHT_COL,
    maxIter     = 20,     # try to improve 20 times
    regParam    = 0.01,   # small penalty to prevent overfitting
)

# Pipeline: indexers → assembler → scaler → LR
lr_pipeline = Pipeline(stages=indexers + [assembler, scaler, lr])

# Evaluators — reused for all models
auc_evaluator = BinaryClassificationEvaluator(
    labelCol   = LABEL_COL,
    metricName = 'areaUnderROC'
)
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol   = LABEL_COL,
    metricName = 'f1'
)

with mlflow.start_run(run_name='logistic_regression') as run:
    lr_run_id = run.info.run_id

    lr_model = lr_pipeline.fit(df_train)   # learn from train set
    lr_preds = lr_model.transform(df_test) # predict on test set

    lr_auc = auc_evaluator.evaluate(lr_preds)
    lr_f1  = f1_evaluator.evaluate(lr_preds)

    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('max_iter',   20)
    mlflow.log_param('reg_param',  0.01)
    mlflow.log_param('split_date', '2026-01-01')
    mlflow.log_param('train_rows', df_train.count())
    mlflow.log_param('test_rows',  df_test.count())
    mlflow.log_metric('auc', lr_auc)
    mlflow.log_metric('f1',  lr_f1)

print(f'  LR AUC : {lr_auc:.4f}')
print(f'  LR F1  : {lr_f1:.4f}')


In [0]:
print('Training Random Forest...')
print()
# ── Why single combo? ─────────────────────────────────────────
# Serverless 1GB ML cache limit — multiple RF models in same session
# causes cache overflow even with del + gc.collect()
# We already know the winner from previous evaluation runs:
#   rf_trees50_depth5   AUC=0.9734  F1=0.9942
#   rf_trees50_depth10  AUC=0.9927  F1=0.9941  ← winner
# Training the winner directly — 1 fit, 1 model in cache, no overflow
# ─────────────────────────────────────────────────────────────

# RF does NOT need StandardScaler — trees are scale-invariant
# Use features directly from assembler — skip the scaler step
assembler_rf = VectorAssembler(
    inputCols    = ALL_INPUT_COLS,
    outputCol    = 'features',
    handleInvalid= 'skip'
)

# Known best params from evaluation runs
best_rf_params = {'numTrees': 50, 'maxDepth': 10}

rf = RandomForestClassifier(
    featuresCol = 'features',
    labelCol    = LABEL_COL,
    weightCol   = WEIGHT_COL,
    numTrees    = best_rf_params['numTrees'],
    maxDepth    = best_rf_params['maxDepth'],
    seed        = 42,
)

rf_pipeline = Pipeline(stages=indexers + [assembler_rf, rf])

with mlflow.start_run(run_name='rf_trees50_depth10') as run:
    best_rf_model = rf_pipeline.fit(df_train)
    best_rf_preds = best_rf_model.transform(df_test)

    best_rf_auc_verify = auc_evaluator.evaluate(best_rf_preds)
    best_rf_f1_verify  = f1_evaluator.evaluate(best_rf_preds)

    mlflow.log_param('model_type', 'RandomForest')
    mlflow.log_param('num_trees',  best_rf_params['numTrees'])
    mlflow.log_param('max_depth',  best_rf_params['maxDepth'])
    mlflow.log_param('split_date', '2026-01-01')
    mlflow.log_metric('auc', best_rf_auc_verify)
    mlflow.log_metric('f1',  best_rf_f1_verify)
    best_rf_run_id = run.info.run_id

print(f'  RF params : {best_rf_params}')
print(f'  RF AUC    : {best_rf_auc_verify:.4f}')
print(f'  RF F1     : {best_rf_f1_verify:.4f}')
print('  best_rf_model ready for Cell 8.')


In [0]:
print('='*55)
print('CHURN MODEL COMPARISON')
print('='*55)
print(f'{"Model":<40} {"AUC":>8} {"F1":>8}')
print('-'*55)
print(f'{"Logistic Regression":<40} {lr_auc:>8.4f} {lr_f1:>8.4f}')
print(f'{"Random Forest (best: " + str(best_rf_params) + ")":<40} {best_rf_auc_verify:>8.4f} {best_rf_f1_verify:>8.4f}')
print('='*55)

# Pick best model — RF wins if AUC >= LR
# AUC is the primary metric — measures ability to rank churned vs active
# regardless of class imbalance, which is why we use it over accuracy
if best_rf_auc_verify >= lr_auc:
    BEST_MODEL      = best_rf_model
    BEST_RUN_ID     = best_rf_run_id
    BEST_AUC        = best_rf_auc_verify
    BEST_MODEL_TYPE = f'RandomForest_{best_rf_params}'
    print('\nWinner: Random Forest')
else:
    BEST_MODEL      = lr_model
    BEST_RUN_ID     = lr_run_id
    BEST_AUC        = lr_auc
    BEST_MODEL_TYPE = 'LogisticRegression'
    print('\nWinner: Logistic Regression')

print(f'Best AUC    : {BEST_AUC:.4f}')
print(f'Best run ID : {BEST_RUN_ID}')
print(f'\nCheck Experiments UI: {EXPERIMENT_NAME}')


In [0]:
print('Registering best model to MLflow Model Registry...')

client = MlflowClient()

# Step 1 — Create model name in registry if it doesn't exist yet
# On second run this will fail silently — that's expected, we just add a new version
try:
    client.create_registered_model(
        name        = MODEL_NAME,
        description = 'Predicts customer churn for ZipCab ride-hailing platform.'
    )
    print(f'  Created model: {MODEL_NAME}')
except Exception:
    print(f'  Model already exists: {MODEL_NAME} — adding new version')

# Step 2 — Build model signature
# Unity Catalog requires a signature — what columns go IN and what comes OUT
# infer_signature() figures it out automatically from a small 100-row sample
sample_input  = df_test.select(ALL_FEATURE_COLS).limit(100)
sample_pandas = sample_input.toPandas()

sample_preds  = BEST_MODEL.transform(sample_input)
sample_output = (
    sample_preds
    .withColumn('churn_probability', vector_to_array('probability')[1])
    .select('prediction', 'churn_probability')
    .toPandas()
)

signature = infer_signature(
    model_input  = sample_pandas,
    model_output = sample_output
)
print('  Signature inferred.')
print(signature)

# Step 3 — Log model + register
with mlflow.start_run(run_name='register_churn_model') as reg_run:
    mlflow.log_param('model_type',    BEST_MODEL_TYPE)
    mlflow.log_param('best_auc',      BEST_AUC)
    mlflow.log_param('train_rows',    df_train.count())
    mlflow.log_param('feature_count', len(ALL_INPUT_COLS))
    mlflow.log_metric('auc', BEST_AUC)

    # Feature importance artifact (RF only)
    # Uses tempfile — no hardcoded paths, works on serverless
    if 'RandomForest' in BEST_MODEL_TYPE:
        rf_stage = BEST_MODEL.stages[-1]
        fi_df = pd.DataFrame({
            'feature':    ALL_INPUT_COLS,
            'importance': rf_stage.featureImportances.toArray()
        }).sort_values('importance', ascending=False)
        with tempfile.TemporaryDirectory() as tmp:
            fi_path = os.path.join(tmp, 'churn_feature_importance.csv')
            fi_df.to_csv(fi_path, index=False)
            mlflow.log_artifact(fi_path)
        print('  Feature importance logged.')

    # Save and register model
    # dfs_tmpdir — Volume path for MLflow to stage model files during save
    # input_example — 5 sample rows so anyone loading knows what format to feed it
    mlflow.spark.log_model(
        spark_model   = BEST_MODEL,
        artifact_path = 'churn_model',
        signature     = signature,
        input_example = sample_pandas.head(5),
        dfs_tmpdir    = TMP_DIR
    )

    model_uri = f'runs:/{reg_run.info.run_id}/churn_model'
    mv = mlflow.register_model(model_uri, MODEL_NAME)

# Step 4 — Add version description
# Visible in Models UI — anyone who opens the model sees what it is
versions      = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max([int(v.version) for v in versions])

client.update_model_version(
    name        = MODEL_NAME,
    version     = str(latest_version),
    description = (
        f'{BEST_MODEL_TYPE}. AUC={BEST_AUC:.4f}. '
        f'Trained on ZipCab 6 months data. 89K customers. '
        f'Split: trained before {SPLIT_DATE}. '
        f'Key feature: days_since_last_trip.'
    )
)
print(f'  Description added to {MODEL_NAME} v{latest_version}')

print()
print(f'  Model registered : {MODEL_NAME}')
print(f'  Version          : {latest_version}')
print(f'  Load URI         : models:/{MODEL_NAME}/{latest_version}')
print(f'  Check Models UI  : {MODEL_NAME}')


In [0]:
print('Saving churn predictions to Delta table...')

# Score all 89K customers — not just train/test split
# Business needs churn probability for every customer
df_all    = df.withColumn(WEIGHT_COL, lit(1.0))
df_scored = BEST_MODEL.transform(df_all)

# Build predictions table
# vector_to_array('probability')[1] — extracts churn probability
# probability vector = [prob_active, prob_churned] — we want index [1]
df_predictions = df_scored.select(
    col('customer_id'),
    col('city'),
    col('is_churned'),
    col(LABEL_COL).alias('actual_label'),
    col('prediction').alias('predicted_label'),
    vector_to_array('probability')[1].alias('churn_probability'),

    # Risk tier — actionable for business teams
    # high_risk   → immediate retention action needed
    # medium_risk → monitor closely, send promo
    # low_risk    → no action needed
    when(vector_to_array('probability')[1] >= 0.7, 'high_risk')
    .when(vector_to_array('probability')[1] >= 0.4, 'medium_risk')
    .otherwise('low_risk').alias('churn_risk_tier'),

).withColumn('_processed_at',  current_timestamp()) \
 .withColumn('_model_version', lit(str(mv.version))) \
 .withColumn('_model_name',    lit(MODEL_NAME))

# Write to Gold layer
df_predictions.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD}.churn_predictions')

pred_count = df_predictions.count()
print(f'  Predictions written : {pred_count:,}')
print(f'  Table               : {GOLD}.churn_predictions')


In [0]:
print('=== CHURN MODEL VERIFICATION ===')
print()

df_verify = spark.table(f'{GOLD}.churn_predictions')
total     = df_verify.count()

print(f'  Total predictions : {total:,}')
print(f'  Model             : {BEST_MODEL_TYPE}')
print(f'  Best AUC          : {BEST_AUC:.4f}')
print()

# Risk tier distribution
print('Risk tier distribution:')
df_verify.groupBy('churn_risk_tier').count().orderBy('count', ascending=False).show()

# Confusion matrix — predicted vs actual
# Good model: most churned customers predicted as churned (label=1, predicted=1)
print('Prediction vs actual (confusion matrix):')
df_verify.groupBy('actual_label', 'predicted_label') \
    .count().orderBy('actual_label', 'predicted_label').show()

# High risk by city — shows which cities need most retention effort
print('High risk customers by city:')
df_verify.filter(col('churn_risk_tier') == 'high_risk') \
    .groupBy('city').count().orderBy('count', ascending=False).show()

# Key validation — churned customers should have much higher avg probability than active
# If both are similar the model is not learning correctly
print('Avg churn probability by actual churn:')
from pyspark.sql.functions import avg, round as spark_round
df_verify.groupBy('is_churned').agg(
    spark_round(avg('churn_probability'), 4).alias('avg_churn_probability')
).orderBy('is_churned').show()

# Feature importance — RF only
# days_since_last_trip should be top feature — confirms model learned correctly
if 'RandomForest' in BEST_MODEL_TYPE:
    print('Top 10 most important features:')
    rf_stage = BEST_MODEL.stages[-1]
    fi = sorted(
        zip(ALL_INPUT_COLS, rf_stage.featureImportances.toArray()),
        key=lambda x: x[1],
        reverse=True
    )
    print(f'  {"Feature":<35} {"Importance":>10}')
    print(f'  {"-"*47}')
    for feat, imp in fi[:10]:
        print(f'  {feat:<35} {imp:>10.4f}')

print()
print('Sample high risk customers:')
df_verify.filter(col('churn_risk_tier') == 'high_risk').select(
    'customer_id', 'city', 'is_churned',
    'churn_probability', 'churn_risk_tier'
).limit(5).show(truncate=False)

print()
print('='*55)
print('Churn model complete.')
print(f'Experiments UI : {EXPERIMENT_NAME}')
print(f'Models UI      : {MODEL_NAME}')
print(f'Predictions    : {GOLD}.churn_predictions')
print('Next           : UrbanRide_12 — Fraud Detection Model')
print('='*55)
